#### 1. Setup the VSCode kernel to use our Python virtual environment
- Click on the ```Select Kernel``` icon in top right of VSCode and select ```Python Environments...```
- The venv environment with the star next to it is the right choice
- This is the python environment where Control-M's Python Client was installed

#### 2. Import the necessary modules from the ctm_python_client
- Hover your mouse to the left side of cell below to invoke the ```Execute Cell``` command for that cell.
- Since this is the first time running code cells, VSCode will automatically install ipykernel as seen bottom right first.
- Upons successful completion of the code cell you wil have a green check mark in the bottom left of cell.

In [ ]:
from ctm_python_client.core.workflow import *
from ctm_python_client.core.comm import *
from ctm_python_client.core.credential import *
from aapi import *

 #### 3. Initialize a ```Workflow``` object.
- Its sort of like a Workspace for adding folders and jobs.
- Here you are establishing access to the Control-M environment
- You may also setup any folder/job defaults like the run_as

In [ ]:
workflow = Workflow(
    Environment.create_saas('https://control-m:8443/automation-api', api_key='${api_key}'),
    WorkflowDefaults(
        controlm_server='instruqt',
        run_as='spark_ctm'
    )
)

#### 4. Create a workload to run across the Agentless Hosts

In [ ]:
# Number of Agentless Hosts and Jobs per each
agentless_count = 5
jobs_per_agent = 100

# The outer loop spans agents to create a Folder for each
for agent_num in range(1,agentless_count+1):
    # The inner loop uses a python list comprehension
    # It creates a list of Cyclic OS Jobs for each Folder
    workload_jobs = [
        JobCommand(f"job_{job_num:04d}",
                host        = f"agentless{agent_num}",
                command     = "env && sleep 10",
                rerun       = Job.Rerun(every="1", units="Minutes", times="99")
                )
        for job_num in range(1,jobs_per_agent)
    ]
    workflow.add(Folder(f"AGENTLESS{agent_num}_JOBS", job_list=workload_jobs))

#### 5. Dump the JobsAsCode to json (Optionally review them)

In [ ]:
with open("farm_jobs.json", "w") as f:
    workflow.dump_json(f, indent=4)

#### 6. Run the Workload

In [ ]:
run = workflow.run()